# <center>[⚔️ Блендинг - смешай и скор вырастет](https://stepik.org/lesson/825510/)</center>

### Оглавление ноутбука

<img src='../images/blending.jpg' align="right" width="750" height="750" />
<br>

<p><font size="3" face="Arial" font-size="large"><ul type="square">
    
<li><a href="#c1">😺🚀 Обучаем CatBoost </a></li>
<li><a href="#c1">🦄🎳 Обучаем LightGBM </a></li>
<li><a href="#c1">👽🔱 Обучаем XGBoost </a></li>
<li><a href="#c1">🐲 Блендинг и принципы блендинга </a></li>
<li><a href="#6">🧸 Выводы и заключения</a>

</li></ul></font></p>

    

<div class="alert alert-info">

* Основная идея данной техники заключается в том, чтобы взять от каждого алгоритма лучшее и совместить несколько разных ML моделей в одну. 
* За счет такого объединения увеличивается обобщающая способность финальной модели и качество улучшается.
* Помимо этого ваша модель становится более стабильной, что позволяет не слететь на приватном лидерборде.
* Особенно хорошо накидывает блендинг, если смешиваемые **модели имеют разную природу**: например, нейронные сети, KNN и решающие деревья, в этом случае они выучивают разные зависимости и хорошо дополняют друг друга.

## Импортируем библиотеки

In [155]:
# Модели для смешивания
import lightgbm as lgbm
import xgboost as xgb
import catboost as cb

In [156]:
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

## Считываем данные

In [157]:
data = pd.read_csv("../data/quickstart_train.csv")

### Заменим категориальные признаки на числовые значения
cat_cols = ["model", "car_type", "fuel_type"]
for col in cat_cols:
    data[col] = data[col].replace(np.unique(data[col]), np.arange(data[col].nunique()))
    data[col] = data[col].astype("category")

data.head()

,car_id,model,car_type,fuel_type,car_rating,year_to_start,riders,year_to_work,target_reg,target_class,mean_rating,distance_sum,rating_min,speed_max,user_ride_quality_median,deviation_normal_count,user_uniq
0,y13744087j,8,1,1,3.78,2015,76163,2021,108.53,another_bug,4.737759,1.214131e+07,0.1,180.855726,0.023174,174,170
1,O41613818T,23,1,1,3.90,2015,78218,2021,35.20,electro_bug,4.480517,1.803909e+07,0.0,187.862734,12.306011,174,174
2,d-2109686j,16,3,1,6.30,2012,23340,2017,38.62,gear_stick,4.768391,1.588366e+07,0.1,102.382857,2.513319,174,173
3,u29695600e,12,0,1,4.04,2011,1263,2020,30.34,engine_fuel,3.880920,1.651883e+07,0.1,172.793237,-5.029476,174,170
4,N-8915870N,16,3,1,4.70,2012,26428,2017,30.45,engine_fuel,4.181149,1.398317e+07,0.1,203.462289,-14.260456,174,171


### Разделим выборку на валидационную и обучающую

In [158]:
cols2drop = ["car_id", "target_reg", "target_class"]

X_train, X_val, y_train, y_val = train_test_split(
    data.drop(cols2drop, axis=1),
    data["target_reg"],
    test_size=0.25,
    stratify=data["target_class"],
    random_state=42,
)
print(X_train.shape, X_val.shape)

(1752, 14) (585, 14)


# <center> Обучим три модели для смешивания - блендинга

## 😺🚀 Обучаем **`CatBoost`**

In [159]:
params_cat = {
    "n_estimators": 1500,
    "learning_rate": 0.03,
    "depth": 3,
    "use_best_model": True,
    "cat_features": cat_cols,
    "text_features": [],
    # 'train_dir' : '/path/to/catboost/model',
    "border_count": 64,
    "l2_leaf_reg": 1,
    "bagging_temperature": 2,
    "rsm": 0.5,
    "loss_function": "RMSE",  # Не определена для регрессии
    # 'auto_class_weights' : 'Balanced', # Не определен для регрессии
    "random_state": 42,
    "custom_metric": ["MAE", "MAPE"],
}

cat_model = cb.CatBoostRegressor(**params_cat)

In [160]:
cat_model.fit(
    X_train,
    y_train,
    verbose=100,
    eval_set=(X_val, y_val),
    early_stopping_rounds=150,
)

0:	learn: 17.3906345	test: 17.7985532	best: 17.7985532 (0)	total: 926us	remaining: 1.39s
100:	learn: 12.0403851	test: 12.3152085	best: 12.3152085 (100)	total: 371ms	remaining: 5.14s
200:	learn: 11.4088372	test: 11.7734917	best: 11.7734917 (200)	total: 724ms	remaining: 4.68s
300:	learn: 11.0615383	test: 11.5972845	best: 11.5972845 (300)	total: 1.07s	remaining: 4.27s
400:	learn: 10.8223003	test: 11.5498790	best: 11.5498790 (400)	total: 1.41s	remaining: 3.86s
500:	learn: 10.6376836	test: 11.5293225	best: 11.5258544 (495)	total: 1.75s	remaining: 3.49s
600:	learn: 10.4762744	test: 11.5542883	best: 11.5258544 (495)	total: 2.1s	remaining: 3.14s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 11.52585441
bestIteration = 495

Shrink model to first 496 iterations.


In [161]:
print('MSE (catboost) : ', np.round(mean_squared_error(cat_model.predict(X_val), y_val), 3))

MSE (catboost) :  132.845


In [162]:
# Сравним с бейзлайном в виде среднего значения
np.round(mean_squared_error(np.ones(len(y_val)) * y_val.mean(), y_val), 3)

319.442

In [163]:
submit = pd.DataFrame({"target": cat_model.predict(X_val).reshape(-1)})
submit.to_csv("../data/catboost_preds.csv", index=False)
submit.head()

,target
0,32.432478
1,47.259392
2,34.470055
3,62.628805
4,72.203130


## 🦄🎳 Обучаем `LightGBM`

In [164]:
params_lgbm = {
    "num_leaves": 200,
    "n_estimators": 1500,
    # "max_depth": 7,
    "min_child_samples": 2073,
    "learning_rate": 0.0051,
    "min_data_in_leaf": 10,
    "feature_fraction": 0.99,
    "categorical_feature": cat_cols,
    'reg_alpha' : 5.0,
    'reg_lambda' : 5.0,
}

lgbm_model = lgbm.LGBMRegressor(**params_lgbm, early_stopping_rounds=100, verbose=150)

In [165]:
lgbm_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    # early_stopping_rounds=100,
    # verbose=150
)


[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=2073 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] feature_fraction is set=0.99, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.99
[LightGBM] [Warning] early_stopping_round is set=100, early_stopping_rounds=100 will be ignored. Current value: early_stopping_round=100
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=2073 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] feature_fraction is set=0.99, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.99
[LightGBM] [Warning] categorical_feature is set=model,car_type,fuel_type, categorical_column=0,1,2 will be ignored. Current value: categorical_feature=model,car_type,fuel_type
[LightGBM] [Debug] Dataset::GetMultiBinFromSparseFeatures: sparse rate 0.865297
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.137601
[LightGBM] [Debug] init for co

LGBMRegressor(categorical_feature=['model', 'car_type', 'fuel_type'],
              early_stopping_rounds=100, feature_fraction=0.99,
              learning_rate=0.0051, min_child_samples=2073, min_data_in_leaf=10,
              n_estimators=1500, num_leaves=200, reg_alpha=5.0, reg_lambda=5.0,
              verbose=150)

In [166]:
print('MSE (lgb) : ', np.round(mean_squared_error(lgbm_model.predict(X_val), y_val), 3))

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=2073 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] feature_fraction is set=0.99, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.99
MSE (lgb) :  141.312


In [167]:
submit = pd.DataFrame({"target": lgbm_model.predict(X_val).reshape(-1)})
submit.to_csv("../data/lgbm_preds.csv", index=False)
submit.head()

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=2073 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] feature_fraction is set=0.99, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.99


,target
0,31.485900
1,43.973935
2,33.595044
3,61.806199
4,67.997354


## 👽🔱 Обучаем `XGBoost`

In [168]:
# xgboost не умеет работать с категориальными признаками, так что нужно сделать ohe
X_train = pd.get_dummies(X_train, columns=["car_type", "fuel_type", "model"])
X_val = pd.get_dummies(X_val, columns=["car_type", "fuel_type", "model"])
X_train.shape

(1752, 43)

In [169]:
params_xgb = {
    "eta": 0.05,
    "max_depth": 5,
    "subsample": 0.7,
    "colsample_bytree": 0.7,
    'gamma': .01,
    'reg_lambda' : 0.1,
    'reg_alpha' : 0.5,
    "objective": "reg:linear",
    "eval_metric": "mae",
    'tree_method' : 'hist', # Supported tree methods for cat fs are `gpu_hist`, `approx`, and `hist`.
    'enable_categorical' : True
    
}

xgb_model = xgb.XGBRegressor(**params_xgb, early_stopping_rounds=100)

In [170]:
xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=25,
)
print('best_iteration', xgb_model.best_iteration)

[0]	validation_0-mae:13.34854	validation_1-mae:13.53848
[25]	validation_0-mae:8.98760	validation_1-mae:9.61713
[50]	validation_0-mae:7.70068	validation_1-mae:8.95202
[75]	validation_0-mae:6.93752	validation_1-mae:8.73708
[99]	validation_0-mae:6.49676	validation_1-mae:8.73955
best_iteration 81


In [171]:
print('MSE (xgboost) : ', np.round(mean_squared_error(xgb_model.predict(X_val), y_val), 3))

MSE (xgboost) :  135.48


In [172]:
submit = pd.DataFrame({"target": xgb_model.predict(X_val).reshape(-1)})
submit.to_csv("../data/xgb_preds.csv", index=False)
submit.head()

,target
0,32.231724
1,43.900486
2,34.952229
3,60.312416
4,70.975258


# <center> 😈 А теперь блендим модели!

* __Быстрый и простой способ сбледнить - это усреднить ответы!__ <br>
* __Но мы сразу покажем смешивание с весами моделей - взвешивание__

In [173]:
cb_model  = pd.read_csv("../data/catboost_preds.csv")["target"]  # 132.051 Лучшая (ставим ей больший вес)
xgb_model = pd.read_csv("../data/xgb_preds.csv")["target"]       # 137.844 Средняя 
lgb_model = pd.read_csv("../data/lgbm_preds.csv")["target"]      # 139.132 Худшая (ставим ей меньший вес)

In [174]:
# score1 > score2 > score3 : w1 > w2 > w3
ensemble = cb_model * 0.50 + xgb_model * 0.35 + lgb_model * 0.15 

In [175]:
print('MSE (ensemble) :', np.round(mean_squared_error(ensemble, y_val), 3))

MSE (ensemble) : 132.236


In [176]:
round((100*(132.051 - 131.080)/132.051), 1)

0.7

# <center> 🥳  Ура, скор улучшился! 
    
<div class="alert alert-info">
    
**Как подбирать веса?**
- Опираясь на скоры на лидерборде
- По локальной валидации
- Ставить веса пропорциональны скору

### Если хочется кодом, то вот более универсальный способ

In [177]:
weights = {"catboost": 0.5, "lgbm": 0.15, "xgb": 0.35}

In [178]:
import os

preds = pd.DataFrame()

# соберем единый датафрейм из наших предсказаний
for model_name in ["catboost", "lgbm", "xgb"]:
    
    path = os.path.join("../data/", f"{model_name}_preds.csv")
    now = pd.read_csv(path).reset_index()

    now["model"] = model_name
    now["target"] *= weights[model_name]
    preds = pd.concat([preds, now])

preds.head()

,index,target,model
0,0,16.216239,catboost
1,1,23.629696,catboost
2,2,17.235027,catboost
3,3,31.314402,catboost
4,4,36.101565,catboost


In [179]:
preds["model"].unique()

array(['catboost', 'lgbm', 'xgb'], dtype=object)

In [180]:
ensemble = preds.groupby("index")["target"].agg("sum")
mean_squared_error(ensemble, y_val)

132.23580062327605

In [ ]:
import numpy as np
n = 3
s = '0.3 0.5 0.4'.split()
scores = []
for i in range(n):
    scores.append(float(s[i]))

## Your code here ...
weights = np.round(np.array([scores]) / np.sum(scores), 7)
weights

array([[0.25     , 0.4166667, 0.3333333]])

# 🐲 Принципы блендинга (просто и логично)

<div class="alert alert-info">

* 🦑 Не бленди, пока не выжал максимум из моделей по отдельности
* 🐳 Чем различнее и сильнее модели, тем эффективнее блендинг (⚠️ эффективность)
* 🐙 При равной точности, ансабль побеждает соло-модель на привате (⚠️ стабильность)
* 🦐 Чем раньше проверишь эффект от блендинга, тем эффективне будет стратегия
* 🐠 Блендить можно с разными весами, пропорционально скорам или разным фичам
* 🐋 Блендинг по фолдам и чекпоинтам обучения - это тоже блендинг
* 🐡 Блендинг по сидам - это стабилизирующий блендинг
* 🐬 Против блендинга только больший блендинг

<div class="alert alert-info">
    
__забегая вперёд__
    
* 🦈 Стекинг сильнее блендинга, но капризнее
* 🎣 Стекать можно с фичами 
* 🎏 Стекинг бывает разных уровней 
* 🐉 Не можешь больше стекать - блендись с решением сокомандников

# 🧸 Выводы и заключения

<div class="alert alert-info">

* Блендинг - это сильный инструмент, который зачастую неплохо поднимает качество моделей.
* При этом само смешивание провести можно вообще в одну строчку, просто взяв среднее моделей и взвесив
* `ensemble = model1 * w1 + model2 * w2 + model3 * w3 + ...`